In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
import lightgbm
from lightgbm import early_stopping, log_evaluation
import xgboost
import joblib
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
N_SPLITS    = 5
RANDOM_STATE = 523
SUBMISSION_PATH = r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Projects\Codes\submission_files\Irrigation_Prediction_KFold.csv'
MODEL_PATH  = r'D:\Codes\Artificial_Intelligence\Machine_Learning\Machine-Learning-With-Scikit-Learn\Projects\Codes\submission_files\final_model.pkl'

MAPPER = {0: 'Low', 1: 'Medium', 2: 'High'}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# ─────────────────────────────────────────────
# MODEL DEFINITIONS
# ─────────────────────────────────────────────
lgbm_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1,
}

xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'n_jobs': -1,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 100,
    'random_state': RANDOM_STATE,
}

rf_params = {
    'n_estimators': 200,
    'max_depth': 7,
    'min_samples_split': 7,
    'min_samples_leaf': 4,
    'max_features': 0.8,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

# ─────────────────────────────────────────────
# K-FOLD RUNNER
# ─────────────────────────────────────────────
def run_kfold(name, model_fn, X, Y, X_test, fit_kwargs_fn=None):
    """
    Runs StratifiedKFold, collects OOF predictions + test fold predictions.
    model_fn      : callable that returns a fresh model instance each fold
    fit_kwargs_fn : callable(model, X_tr, Y_tr, X_val, Y_val) → dict of extra fit kwargs
    Returns       : oof_preds (N,), test_preds (N_test,), fold_metrics list
    """
    oof_preds   = np.zeros(len(X), dtype=int)
    test_preds  = np.zeros((len(X_test), N_SPLITS), dtype=int)
    fold_metrics = []

    print(f"\n{'='*55}")
    print(f"  {name}  —  {N_SPLITS}-Fold Cross Validation")
    print(f"{'='*55}")

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, Y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        Y_tr, Y_val = Y.iloc[tr_idx], Y.iloc[val_idx]

        model = model_fn()
        extra = fit_kwargs_fn(model, X_tr, Y_tr, X_val, Y_val) if fit_kwargs_fn else {}
        model.fit(X_tr, Y_tr, **extra)

        val_pred  = model.predict(X_val)
        oof_preds[val_idx] = val_pred

        test_preds[:, fold - 1] = model.predict(X_test)

        acc = accuracy_score(Y_val, val_pred)
        f1  = f1_score(Y_val, val_pred, average='weighted')
        fold_metrics.append({'fold': fold, 'acc': acc, 'f1': f1})
        print(f"  Fold {fold}  →  Acc: {acc:.4f}  |  F1: {f1:.4f}")

    oof_acc = accuracy_score(Y, oof_preds)
    oof_f1  = f1_score(Y, oof_preds, average='weighted')
    print(f"\n  OOF Acc : {oof_acc:.4f}  |  OOF F1: {oof_f1:.4f}")

    # majority vote across folds for test
    from scipy import stats
    final_test_preds, _ = stats.mode(test_preds, axis=1)
    final_test_preds = final_test_preds.flatten().astype(int)

    return oof_preds, final_test_preds, fold_metrics, oof_f1


# ─────────────────────────────────────────────
# LGBM FIT KWARGS (early stopping needs eval_set)
# ─────────────────────────────────────────────
def lgbm_fit_kwargs(model, X_tr, Y_tr, X_val, Y_val):
    return {
        'eval_set': [(X_val, Y_val)],
        'callbacks': [early_stopping(100, verbose=False), log_evaluation(period=0)],
    }

def xgb_fit_kwargs(model, X_tr, Y_tr, X_val, Y_val):
    return {
        'eval_set': [(X_val, Y_val)],
        'verbose': False,
    }


# ─────────────────────────────────────────────
# RUN ALL THREE MODELS
# ─────────────────────────────────────────────
# Combine train + val back into a single frame for K-Fold
X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
Y_full = pd.concat([Y_train, Y_val], axis=0).reset_index(drop=True)

lgbm_oof, lgbm_test, lgbm_metrics, lgbm_f1 = run_kfold(
    "LightGBM",
    lambda: lightgbm.LGBMClassifier(**lgbm_params),
    X_full, Y_full, test_df,
    fit_kwargs_fn=lgbm_fit_kwargs
)

xgb_oof, xgb_test, xgb_metrics, xgb_f1 = run_kfold(
    "XGBoost",
    lambda: xgboost.XGBClassifier(**xgb_params),
    X_full, Y_full, test_df,
    fit_kwargs_fn=xgb_fit_kwargs
)

rf_oof, rf_test, rf_metrics, rf_f1 = run_kfold(
    "Random Forest",
    lambda: RandomForestClassifier(**rf_params),
    X_full, Y_full, test_df,
)


# ─────────────────────────────────────────────
# ENSEMBLE  —  weighted majority vote on test preds
# ─────────────────────────────────────────────
print(f"\n{'='*55}")
print("  OOF F1 scores (used as ensemble weights)")
print(f"  LGBM  : {lgbm_f1:.4f}")
print(f"  XGB   : {xgb_f1:.4f}")
print(f"  RF    : {rf_f1:.4f}")

# Stack test predictions and pick the model with highest OOF F1
all_test = np.stack([lgbm_test, xgb_test, rf_test], axis=1)   # (N_test, 3)
weights  = np.array([lgbm_f1, xgb_f1, rf_f1])
best_idx = np.argmax(weights)
names    = ["LightGBM", "XGBoost", "Random Forest"]

# Weighted majority vote
from scipy import stats
final_test_pred, _ = stats.mode(all_test, axis=1)
final_test_pred = final_test_pred.flatten().astype(int)

print(f"\n  Best single model : {names[best_idx]}  (F1={weights[best_idx]:.4f})")
print(f"  Final test preds use weighted majority vote across all three.")


# ─────────────────────────────────────────────
# FINAL MODEL  —  retrain best model on ALL data
# ─────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Retraining {names[best_idx]} on full dataset …")

if best_idx == 0:
    final_model = lightgbm.LGBMClassifier(**{**lgbm_params, 'n_estimators': 500})
    final_model.fit(X_full, Y_full)
elif best_idx == 1:
    final_model = xgboost.XGBClassifier(**{k: v for k, v in xgb_params.items()
                                           if k != 'early_stopping_rounds'})
    final_model.fit(X_full, Y_full)
else:
    final_model = RandomForestClassifier(**rf_params)
    final_model.fit(X_full, Y_full)

joblib.dump(final_model, MODEL_PATH)
print(f"  Model saved → {MODEL_PATH}")


# ─────────────────────────────────────────────
# SUBMISSION
# ─────────────────────────────────────────────
submission = pd.DataFrame({
    'id': test_id.reset_index(drop=True),
    'Irrigation_Need': pd.Series(final_test_pred).map(MAPPER)
})

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"\n  Submission saved → {SUBMISSION_PATH}")
print(f"  Shape : {submission.shape}")
print(submission.head(10))
print("\nLabel distribution:")
print(submission['Irrigation_Need'].value_counts())